# `0936` 앞발 행동 횟수: 고정 기준·후보 합집합·상태 흐름

질문은 “강아지가 유리문 앞에서 앞발을 몇 번 들었다 내렸어? 뒷발이 바닥에 남아도 포함해.”입니다. 이 실험에서는 **같은 강아지의 두 앞발이 함께 내려간 상태 → 문 쪽으로 들린 상태**를 1회로 정의했습니다. 뒷발이 바닥에 남아도 포함합니다. 같은 자세를 유지하는 동안은 다시 세지 않습니다.

**앞선 0.5초·0.25초 비교에는 질문 해석이 달라지는 문제가 있었습니다.** 앞선 결과는 아래 표에서 제외하고, 고정한 `smartbench_0936-fixed-intent.json`으로 두 간격을 다시 실행했습니다. `gpt-4o`를 각각 한 번 실행했으므로 프레임 간격의 순수한 인과 효과를 추정한 실험은 아닙니다. 사람 기준 총 8회는 사용자 판단이고, 개별 사건 시각은 프레임 잠정 검토입니다.

## 결과

| 방법 | 살핀 프레임 | 후보가 있는 사건 | 전·후 검증 통과 | 최종 확정 | 미확정 |
|---|---:|---:|---:|---:|---:|
| 0.5초 탐색 | 64 | 7/8 | 6/8 | 6 | 1 |
| 0.25초 탐색 | 122 | 7/8 | 6/8 | 6 | 0 |
| 두 후보 목록 합집합 | 두 탐색 결과 재사용 | 8/8 | 7/8 | **7** | **1** |
| 0.25초 프레임별 상태 흐름 | 87 | 6/8* | 3/8 | 3 | 2 |

\* 상태 흐름 방식은 7개 전환을 제안했지만, 잠정 사건 시각 ±0.25초와 맞는 전환은 6개였습니다. 모델이 제시한 전환을 두 핵심 프레임으로 독립 검증하면 확정은 3개였습니다. 숫자만 맞춘 결과가 아니라 아래 사건별 근거를 보세요.

## 사람 기준 8개 사건별 결과

| 사건 | 잠정 시각(초) | 0.5초 | 0.25초 | 후보 합집합 | 프레임별 상태 |
|---:|---:|---|---|---|---|
| 1 | 0.25~0.85 | 미확정 (0.5초) | 검증 거절 (0.25초) | 미확정 (0.375초) | 후보 없음 |
| 2 | 1.50~2.25 | 확정 (2.0초) | 후보 없음 | 확정 (2.0초) | 확정 (1.5초) |
| 3 | 4.25~4.90 | 확정 (4.5, 4.75초) | 확정 (4.5초) | 확정 (4.583초) | 확정 (4.5초) |
| 4 | 6.00~6.75 | 확정 (6.25초) | 확정 (6.25초) | 확정 (6.25초) | 확정 (6.25초) |
| 5 | 7.40~8.00 | 확정 (7.75초) | 확정 (7.5초) | 확정 (7.625초) | 검증 거절 (7.25초) |
| 6 | 9.00~10.00 | 확정 (9.25초) | 확정 (9.5초) | 확정 (9.375초) | 미확정 (9.25초) |
| 7 | 15.25~16.00 | 확정 (15.75초) | 확정 (15.5, 16.0초) | 확정 (15.75초) | 미확정 (15.25초) |
| 8 | 19.00~19.75 | 후보 없음 | 확정 (19.25초) | 확정 (19.25초) | 후보 없음 |

`후보 없음`은 탐색 실패, `검증 거절`은 후보를 찾았으나 전·후 상태 검사가 실패한 경우입니다. `미확정`은 횟수에 더하지 않습니다. 시각 일치만으로 같은 동물인지 증명한 것은 아닙니다.

In [1]:
import json
from pathlib import Path
root = Path.cwd()
if root.name == "notebooks": root = root.parent
report_path = root / "notebooks/smarthome-bench-evidence/new-pipeline/ablation-0936-fixed/comparison.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
for name, metric in report["metrics"].items():
    print(f'{name}: 후보 사건 {metric["reference_events_with_candidate"]}/8, '
          f'최종 확정 {metric["reference_events_final_confirmed"]}/8, '
          f'미확정 {metric["pipeline_uncertain_count"]}')

0.5: 후보 사건 7/8, 최종 확정 6/8, 미확정 1
0.25: 후보 사건 7/8, 최종 확정 6/8, 미확정 0
union: 후보 사건 8/8, 최종 확정 7/8, 미확정 1
state_timeline: 후보 사건 6/8, 최종 확정 3/8, 미확정 2

In [ ]:
for event in report["events"]:
    print(f'\n사건 {event["event_number"]}: {event["reference_start"]}~{event["reference_end"]}초')
    for name in ("0.5", "0.25", "union", "state_timeline"):
        run = event["runs"][name]
        print(f'  {name}: {run["stage"]}, 후보 {run["candidate_times"]}')
        for detail in run["details"]:
            print('    ', detail["time"], '전·후', detail["verification_status"],
                  '최종', detail["final_decision"], '원인',
                  detail["review_reason"] or detail["evidence_check_reason"])

## 왜 7회에서 멈췄나

첫 사건은 약 0.25~0.85초로 사람 기준에 포함됩니다. 그러나 영상 0초에 이미 앞발이 들린 것으로 보이거나 내려간 자세가 분명하지 않아, **영상 안에서 내려감 → 들림을 확정할 수 없습니다.** 프레임별 상태 모델도 0~0.75초를 `after`, 1.0초부터 `before`로 분류했습니다. 이 분류 역시 모델 판단입니다. 그래서 후보 합집합의 답은 “확실히 7회, 시작 장면 포함 가능 시 8회”입니다. 첫 사건을 자동 확정 8회로 바꾸는 것은 현재 근거로는 과장입니다.

기존 집계는 모델이 한 사건의 `end_time`을 길게 추정하면, 그 사이에 발을 내린 프레임이 확인돼도 다음 사건을 중복으로 버렸습니다. 집계기가 **독립 검증된 리셋 시각**을 우선하게 수정하자 같은 저장 응답의 최종 확정이 0.5초 5→6, 0.25초 3→6, 합집합 4→7로 바뀌었습니다. 모델 응답을 새로 맞춘 결과가 아닙니다.

## 영상 확인

아래 셀은 로컬 `data/videos/smartbench_0936.mp4`가 있으면 재생합니다. 저장된 표와 JSON을 보는 데는 영상이나 API 키가 필요 없습니다. 첫 0~1.25초를 직접 확인하면 경계 사건이 왜 미확정인지 볼 수 있습니다.

In [ ]:
video = root / "data/videos/smartbench_0936.mp4"
if not video.exists():
    print("로컬 영상 없음:", video)
else:
    try:
        from IPython.display import Video, display
    except ImportError:
        print("영상 재생은 Jupyter/IPython에서 열어주세요:", video)
    else:
        display(Video(filename=str(video), embed=False, html_attributes="controls"))

## 재실행 방법

영상과 `.env`의 `OPENAI_API_KEY`가 필요합니다. 아래 검증 명령은 API 비용이 발생합니다. `compare_0936_sampling.py`는 저장된 JSON만 읽습니다.

```bash
.venv/bin/python scripts/validate_video_aggregate.py --ids 0936 --scan-model gpt-4o --verify-model gpt-4o --coarse-step 0.5 --intent-file notebooks/smarthome-bench-evidence/new-pipeline/smartbench_0936-fixed-intent.json --output notebooks/smarthome-bench-evidence/new-pipeline/ablation-0936-fixed/step050
.venv/bin/python scripts/validate_video_aggregate.py --ids 0936 --scan-model gpt-4o --verify-model gpt-4o --coarse-step 0.25 --intent-file notebooks/smarthome-bench-evidence/new-pipeline/smartbench_0936-fixed-intent.json --output notebooks/smarthome-bench-evidence/new-pipeline/ablation-0936-fixed/step025
.venv/bin/python scripts/validate_0936_union.py
.venv/bin/python scripts/validate_0936_state_timeline.py
python3 scripts/compare_0936_sampling.py
```

## 해석과 다음 과제

- **후보 합집합은 이번 영상에서 효과적**이었습니다. 단독 탐색은 각각 7개 사건에 후보를 냈고, 합집합은 8개 모두에 후보를 냈습니다. 다만 다른 영상에서도 같은 이득이 나는지 검증이 필요합니다.
- **프레임별 상태 모델만으로는 부족**했습니다. 7개 전환 제안 중 독립 전·후 프레임 확인을 통과한 것은 3개였습니다. 상태 분류 결과를 정답처럼 취급하지 않고 후보 제안에만 활용해야 합니다.
- **시작 경계와 개체 추적은 여전히 제한**입니다. 이 실험은 한 강아지를 가정했습니다. 여러 강아지가 섞인 영상에는 개체 ID 추적과 사건별 사람 시각 라벨이 추가로 필요합니다.

이 결과는 로컬 노트북 검증이며 운영 `/query` 또는 worker 배포 성능을 뜻하지 않습니다.